In [ ]:
"""
CHI-FIELD PDE SOLVER — Test 1A: Propagation Simulation
========================================================
Simulates the coupled chi-field and Phi-field PDEs:
  Box(chi) + m^2*chi + (lambda/6)*chi^3 + xi*R*chi = J_grace + g_Phi*Phi^2
  Box(Phi) + m_Phi^2*Phi + 2*g_Phi*chi*Phi = J_witness

Goal: Show stable numerical solutions exist for physically plausible parameters.
Vary coupling constants and document stable vs unstable regimes.

David Lowe (POF 2828) | March 2026
"""

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from config import M_CHI, XI_COUPLING, LAMBDA_SELF, BETA_GRACE, EPSILON_UV

In [ ]:
class ChiFieldSolver:
    """
    1+1D PDE solver for coupled chi-Phi system.
    Uses method of lines: discretize space, integrate time with RK4.
    """

    def __init__(self, nx=256, dx=0.1, dt=0.001,
                 m_chi=1.0, m_phi=0.5,
                 lam=0.1, xi=0.01, g_phi=0.05,
                 beta_g=0.3):
        self.nx = nx
        self.dx = dx
        self.dt = dt
        self.m_chi = m_chi
        self.m_phi = m_phi
        self.lam = lam
        self.xi = xi
        self.g_phi = g_phi
        self.beta_g = beta_g

    def laplacian_1d(self, field):
        """Second spatial derivative via central differences."""
        return (jnp.roll(field, 1) + jnp.roll(field, -1) - 2 * field) / self.dx**2

    def grace_source(self, chi, phi, s_local):
        """J_grace = beta_G * Phi * tanh(chi) / (S + eps)"""
        f_reg = jnp.tanh(chi)
        return self.beta_g * phi * f_reg / (s_local + EPSILON_UV)

    def rhs_chi(self, chi, chi_t, phi, s_local):
        """
        d^2(chi)/dt^2 = nabla^2(chi) - m^2*chi - (lam/6)*chi^3 - xi*R*chi + J_grace + g_phi*phi^2

        In flat space R=0, so xi*R*chi term drops out.
        For non-flat: R is a function of chi (backreaction) — future work.
        """
        lap = self.laplacian_1d(chi)
        j_grace = self.grace_source(chi, phi, s_local)
        return (lap
                - self.m_chi**2 * chi
                - (self.lam / 6.0) * chi**3
                + j_grace
                + self.g_phi * phi**2)

    def rhs_phi(self, chi, phi, phi_t):
        """
        d^2(Phi)/dt^2 = nabla^2(Phi) - m_Phi^2*Phi - 2*g_Phi*chi*Phi
        """
        lap = self.laplacian_1d(phi)
        return (lap
                - self.m_phi**2 * phi
                - 2.0 * self.g_phi * chi * phi)

    def initial_conditions(self, key=None):
        """Gaussian bump initial conditions."""
        if key is None:
            key = jax.random.PRNGKey(2828)
        x = jnp.arange(self.nx) * self.dx
        center = self.nx * self.dx / 2.0
        sigma = self.nx * self.dx / 10.0

        chi0 = 0.5 * jnp.exp(-(x - center)**2 / (2 * sigma**2))
        phi0 = 0.1 * jnp.exp(-(x - center)**2 / (2 * sigma**2))
        chi_t0 = jnp.zeros(self.nx)
        phi_t0 = jnp.zeros(self.nx)
        s_local = 0.1 * jnp.ones(self.nx)  # uniform entropy background

        return chi0, chi_t0, phi0, phi_t0, s_local

    def step_rk4(self, state, s_local):
        """One RK4 step for the full [chi, chi_t, phi, phi_t] system."""
        chi, chi_t, phi, phi_t = state

        def deriv(chi_, chi_t_, phi_, phi_t_):
            d_chi = chi_t_
            d_chi_t = self.rhs_chi(chi_, chi_t_, phi_, s_local)
            d_phi = phi_t_
            d_phi_t = self.rhs_phi(chi_, phi_, phi_t_)
            return d_chi, d_chi_t, d_phi, d_phi_t

        dt = self.dt
        k1 = deriv(chi, chi_t, phi, phi_t)
        k2 = deriv(chi + 0.5*dt*k1[0], chi_t + 0.5*dt*k1[1],
                    phi + 0.5*dt*k1[2], phi_t + 0.5*dt*k1[3])
        k3 = deriv(chi + 0.5*dt*k2[0], chi_t + 0.5*dt*k2[1],
                    phi + 0.5*dt*k2[2], phi_t + 0.5*dt*k2[3])
        k4 = deriv(chi + dt*k3[0], chi_t + dt*k3[1],
                    phi + dt*k3[2], phi_t + dt*k3[3])

        chi_new = chi + (dt/6.0)*(k1[0] + 2*k2[0] + 2*k3[0] + k4[0])
        chi_t_new = chi_t + (dt/6.0)*(k1[1] + 2*k2[1] + 2*k3[1] + k4[1])
        phi_new = phi + (dt/6.0)*(k1[2] + 2*k2[2] + 2*k3[2] + k4[2])
        phi_t_new = phi_t + (dt/6.0)*(k1[3] + 2*k2[3] + 2*k3[3] + k4[3])

        return chi_new, chi_t_new, phi_new, phi_t_new

    def run(self, n_steps=1000, save_every=10):
        """
        Run the simulation.

        Returns:
            dict with time series of chi, phi, energies, stability metrics
        """
        chi, chi_t, phi, phi_t, s_local = self.initial_conditions()
        state = (chi, chi_t, phi, phi_t)

        history = {
            "chi_max": [],
            "chi_min": [],
            "phi_max": [],
            "phi_min": [],
            "chi_energy": [],
            "phi_energy": [],
            "total_energy": [],
            "times": [],
            "chi_snapshots": [],
            "phi_snapshots": [],
        }

        for step in range(n_steps):
            state = self.step_rk4(state, s_local)
            chi, chi_t, phi, phi_t = state

            if step % save_every == 0:
                t = step * self.dt
                # Energy density: E = 1/2 chi_t^2 + 1/2 (grad chi)^2 + V(chi)
                grad_chi = (jnp.roll(chi, -1) - jnp.roll(chi, 1)) / (2 * self.dx)
                v_chi = 0.5 * self.m_chi**2 * chi**2 + (self.lam/24.0) * chi**4
                e_chi = float(jnp.sum(0.5*chi_t**2 + 0.5*grad_chi**2 + v_chi) * self.dx)

                grad_phi = (jnp.roll(phi, -1) - jnp.roll(phi, 1)) / (2 * self.dx)
                v_phi = 0.5 * self.m_phi**2 * phi**2
                e_phi = float(jnp.sum(0.5*phi_t**2 + 0.5*grad_phi**2 + v_phi) * self.dx)

                history["times"].append(t)
                history["chi_max"].append(float(jnp.max(chi)))
                history["chi_min"].append(float(jnp.min(chi)))
                history["phi_max"].append(float(jnp.max(phi)))
                history["phi_min"].append(float(jnp.min(phi)))
                history["chi_energy"].append(e_chi)
                history["phi_energy"].append(e_phi)
                history["total_energy"].append(e_chi + e_phi)
                history["chi_snapshots"].append(np.array(chi).tolist())
                history["phi_snapshots"].append(np.array(phi).tolist())

        # Stability assessment
        energies = np.array(history["total_energy"])
        chi_maxes = np.array(history["chi_max"])

        stable = bool(
            np.all(np.isfinite(energies)) and
            np.all(np.abs(chi_maxes) < 1e6)
        )
        energy_drift = float(np.std(energies) / (np.mean(np.abs(energies)) + 1e-10))

        history["stable"] = stable
        history["energy_drift_pct"] = round(energy_drift * 100, 4)
        history["n_steps"] = n_steps
        history["params"] = {
            "m_chi": self.m_chi,
            "m_phi": self.m_phi,
            "lambda": self.lam,
            "xi": self.xi,
            "g_phi": self.g_phi,
            "beta_g": self.beta_g,
            "nx": self.nx,
            "dx": self.dx,
            "dt": self.dt,
        }

        return history

In [ ]:
def parameter_scan(param_name, values, base_params=None, n_steps=500):
    """
    Scan one parameter across a range and classify stable vs unstable.

    Returns list of (param_value, stable, energy_drift).
    """
    if base_params is None:
        base_params = {
            "nx": 128, "dx": 0.1, "dt": 0.0005,
            "m_chi": 1.0, "m_phi": 0.5,
            "lam": 0.1, "xi": 0.01, "g_phi": 0.05,
            "beta_g": 0.3,
        }

    results = []
    for val in values:
        params = dict(base_params)
        params[param_name] = val
        solver = ChiFieldSolver(**params)
        try:
            history = solver.run(n_steps=n_steps, save_every=50)
            results.append({
                "param_value": float(val),
                "stable": history["stable"],
                "energy_drift_pct": history["energy_drift_pct"],
                "chi_max": float(np.max(history["chi_max"])),
            })
        except Exception as e:
            results.append({
                "param_value": float(val),
                "stable": False,
                "error": str(e),
            })

    return results